#Parte 2: Capa Semántica (M2)

##Ejercicio 2.1: View semántica con JOIN — GUIDED


In [0]:
%sql
-- Primero crear el schema si no existe
CREATE SCHEMA IF NOT EXISTS bootcamp.semantica;

-- La view encapsula el JOIN para que el consumidor no necesite saber el schema
CREATE OR REPLACE VIEW bootcamp.semantica.v_propiedades_venta_usd AS
SELECT 
    dz.partido,
    dz.region,
    fp.precio,
    fp.precio_por_m2,
    fp.metros_cuadrados_totales,
    fp.ambientes
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
JOIN bootcamp.gold.dim_tipo_operacion dto ON fp.tipo_operacion_id = dto.tipo_operacion_id
WHERE dto.tipo_operacion = 'venta' AND fp.moneda = 'USD';

-- Es mejor porque: 1) el analista no necesita saber que hay 3 tablas y 2 JOINs,
-- 2) si el schema cambia, solo se actualiza la view, no todos los queries del analista,
-- 3) actúa como "contrato" entre el DE y el consumidor.

##Ejercicio 2.2: View de métricas agregadas — INDEPENDENT


In [0]:
%sql
CREATE OR REPLACE VIEW bootcamp.semantica.v_comparativa_zonas AS
SELECT 
    dz.partido,
    COUNT(*) as total_propiedades,
    ROUND(AVG(fp.precio), 2) as precio_promedio,
    ROUND(MIN(fp.precio), 2) as precio_minimo,
    ROUND(MAX(fp.precio), 2) as precio_maximo,
    ROUND(AVG(fp.precio_por_m2), 2) as precio_m2_promedio,
    ROUND(AVG(fp.ambientes), 1) as ambientes_promedio
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
JOIN bootcamp.gold.dim_tipo_operacion dto ON fp.tipo_operacion_id = dto.tipo_operacion_id
WHERE dto.tipo_operacion = 'venta' AND fp.moneda = 'USD'
GROUP BY dz.partido;

SELECT * FROM bootcamp.semantica.v_comparativa_zonas ORDER BY total_propiedades DESC LIMIT 15;


##Ejercicio 2.3: Explorar views del catálogo — INDEPENDENT

In [0]:
CREATE OR REPLACE VIEW bootcamp.semantica.v_propiedades_completa AS
SELECT 
    dz.partido,
    dz.region,
    fp.precio,
    fp.moneda,
    fp.precio_por_m2,
    fp.metros_cuadrados_totales,
    fp.ambientes
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
JOIN bootcamp.gold.dim_tipo_operacion dto ON fp.tipo_operacion_id = dto.tipo_operacion_id
WHERE dto.tipo_operacion = 'venta' AND fp.moneda = 'USD';

In [0]:
%sql
-- Listar views
USE CATALOG bootcamp;
SHOW VIEWS IN bootcamp.semantica;

In [0]:
-- DESCRIBE muestra columnas y tipos. EXTENDED agrega metadata (ubicación, propietario, definición SQL).
DESCRIBE EXTENDED bootcamp.semantica.v_propiedades_completa;